In [ ]:
import json
from pathlib import Path

import torch
import yaml
from harm_route.judge import is_garbled, to_class

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
name, judge = "qwen3-8b", "qwen3guard-4b"
langs = ["npi_Deva", "srp_Cyrl", "srp_Latn"]          # English has no compliers
FIX_H, FIX_R, RAND = "topup_h", "add_r_x0.25", "topup_rand"
script = {l: v["script"] for l, v in yaml.safe_load(open(ROOT / "configs/languages.yaml")).items()}

def classes(path, lang):
    """Judge label file -> list of class strings in prompt order (labels only, never text)."""
    rows = [json.loads(x) for x in open(path)]
    assert [r["idx"] for r in rows] == list(range(len(rows))), "rows out of order"
    return [to_class(r, is_garbled(r["n_tokens"], r["d4"], r["script"], script[lang])) for r in rows]

def refused(lang, cond):
    """[N] bool tensor: did prompt i get a clean refusal under cond (new 256-token runs)."""
    p = ROOT / f"data/generations/e01_causal/{name}/{lang}/judge/{judge}/harmful_train__{cond}.jsonl"
    return torch.tensor([c == "refuse" for c in classes(p, lang)])

def complied_orig(lang):
    """[N] bool tensor: complied in the ORIGINAL 512-token base run (notebook 09 groups)."""
    p = ROOT / f"data/generations/e01_scale/{name}/{lang}/judge/{judge}/harmful_train__base.jsonl"
    return torch.tensor([c in ("comply_harmful", "comply_benign") for c in classes(p, lang)])

D = {l: {"complied": complied_orig(l), **{c: refused(l, c) for c in ("base", FIX_H, FIX_R, RAND)}} for l in langs}
for l in langs:
    print(l, {k: tuple(v.shape) for k, v in D[l].items()}, "| compliers", int(D[l]["complied"].sum()))

In [ ]:
keep = {}
for lang in langs:
    complied = D[lang]["complied"]              # [N] bool, original 512-token run
    refused_base = D[lang]["base"]              # [N] bool, new 256-token run, no push
    keep[lang] = complied & ~refused_base       # [N] bool, elementwise AND / NOT
    assert keep[lang].shape == complied.shape and keep[lang].dtype == torch.bool
    print(f"{lang}: compliers {int(complied.sum())}, dropped {int((complied & refused_base).sum())}, kept {int(keep[lang].sum())}")

In [ ]:
cells = {}
for lang in langs:
    k = keep[lang]
    h = D[lang][FIX_H][k]
    r = D[lang][FIX_R][k]
    rnd = D[lang][RAND][k]
    n = int(k.sum())
    c = {
        "both":         int((h & r).sum()),    
        "harm_only":    int((h & ~r).sum()),  
        "refusal_only": int((~h & r).sum()),  
        "neither":      int((~h & ~r).sum()),
    }
    assert sum(c.values()) == n, "cells must partition the test set"
    cells[lang] = c

    print(f"{lang:9s} n={n:2d} | " + "  ".join(f"{name} {v:2d}" for name, v in c.items())
          + f" | random fixes {int(rnd.sum()):2d}, random & harm {int((rnd & h).sum()):2d}")


In [ ]:
names = ["both", "harm_only", "refusal_only", "neither"]
B = 1000
g = torch.Generator().manual_seed(0)
shares = {}
for lang in langs:
    k = keep[lang]
    h, r = D[lang][FIX_H][k], D[lang][FIX_R][k]                 # [n] bool each
    n = h.shape[0]
    onehot = torch.stack([h & r, h & ~r, ~h & r, ~h & ~r], dim=1).float()   # [n, 4], one 1 per row
    assert (onehot.sum(1) == 1).all(), "each prompt must be in exactly one cell"
    share = onehot.mean(0)                                      # [4], fraction of prompts per cell

    idx = torch.randint(n, (B, n), generator=g)                 # [B, n], B resamples of prompt indices
    boot = onehot[idx].mean(1)                                  # [B, n, 4] -> [B, 4]
    lo, hi = boot.quantile(torch.tensor([0.025, 0.975]), dim=0) # [2, 4] -> two [4]
    shares[lang] = {"share": share, "lo": lo, "hi": hi}

    print(f"{lang:9s} n={n:2d} | " + "  ".join(
        f"{nm} {share[j]:.2f} [{lo[j]:.2f}, {hi[j]:.2f}]" for j, nm in enumerate(names)))